# 03 · DataFrames & Schema

> **Module 2 · Spark Fundamentals · Lecture 2.3**

In this notebook you will:

- Read CSV, JSON, and Parquet with `spark.read` and choose between `inferSchema` and an explicit `StructType`
- Persist a schema with `schema.json` and rehydrate it with `StructType.fromJson`
- Select, enrich, and cast columns with `select`, `withColumn`, and `try_cast`
- Move data to the driver deliberately with `toPandas()` and `toLocalIterator()`
- Handle nulls with `dropna`, `fillna`, and `coalesce`

---


## Setup / prerequisites

- **Python 3.9–3.11** and **Java 11 or 17** on your machine.
- **PySpark 3.5.x**: `pip install "pyspark==3.5.1" pandas pyarrow numpy`
- This notebook is **self-contained and offline**: every code cell runs top-to-bottom with
  no GCP account. All data operations use local paths under `/tmp/`.
- GCS / BigQuery / Pub/Sub / JDBC code appears in **clearly-marked optional cells** that
  you uncomment after authenticating with `gcloud auth application-default login`.
- On Dataproc, **do not `pip install pyspark`** — use the Spark runtime already on the
  image (Spark 3.5 on Dataproc 2.2).


## 1 · Generate sample files

The first code cell writes CSV, JSON, and Parquet files under `/tmp/`, so every later
cell has real files to read and the notebook works with no cloud account.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pyspark

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("notebook-01")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("PySpark :", pyspark.__version__)
print("Spark   :", spark.version)
print("Master  :", spark.sparkContext.master)


In [ ]:
import json
import os
import shutil

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType,
)

DATA_DIR = "/tmp/dataproc_notebook_03"
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

# --- CSV with a header, including a deliberately dirty numeric and a blank field ---
csv_rows = [
    {"order_id": "o-1", "customer_id": "c-1", "region": "EMEA", "amount": "100.50", "quantity": "2"},
    {"order_id": "o-2", "customer_id": "c-2", "region": "APAC",  "amount": "80.00",  "quantity": "1"},
    {"order_id": "o-3", "customer_id": "c-1", "region": "EMEA", "amount": "not_a_number", "quantity": "3"},
    {"order_id": "o-4", "customer_id": "c-3", "region": "AMER",  "amount": "",        "quantity": "5"},
    {"order_id": "o-5", "customer_id": "c-2", "region": "APAC",  "amount": "310.75", "quantity": "1"},
]
with open(f"{DATA_DIR}/orders.csv", "w", encoding="utf-8") as fh:
    fh.write("order_id,customer_id,region,amount,quantity\n")
    for row in csv_rows:
        fh.write(",".join(row[k] for k in
                          ["order_id", "customer_id", "region", "amount", "quantity"]) + "\n")

# --- JSON (one object per line) ---
json_rows = [
    {"event_id": "e-1", "event_type": "click",   "user_id": "u-1", "amount": 10.0},
    {"event_id": "e-2", "event_type": "purchase", "user_id": "u-2", "amount": 99.5},
    {"event_id": "e-3", "event_type": "view",    "user_id": "u-1", "amount": 0.0},
]
with open(f"{DATA_DIR}/events.json", "w", encoding="utf-8") as fh:
    for row in json_rows:
        fh.write(json.dumps(row) + "\n")

print("wrote:")
for name in sorted(os.listdir(DATA_DIR)):
    print("  ", name, os.path.getsize(os.path.join(DATA_DIR, name)), "bytes")

## 2 · Reading CSV, JSON, and Parquet

`spark.read` is the unified reader. The shortcuts (`read.csv`, `read.json`,
`read.parquet`) are sugar over `format(...).option(...).load(...)`.

| Format | Schema in file | Read speed | Typical use |
|---|---|---|---|
| CSV | No | Slowest | Interop, small human-readable drops |
| JSON | No (NDJSON) | Slow | Event payloads |
| Parquet | Footer | **Fastest** | Analytical warehouse tables |

In [ ]:
# CSV with inference (exploratory only).
csv_inferred = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{DATA_DIR}/orders.csv")
)
print("== inferred schema ==")
csv_inferred.printSchema()
csv_inferred.show(truncate=False)

# JSON
json_df = spark.read.json(f"{DATA_DIR}/events.json")
print("\n== json schema ==")
json_df.printSchema()
json_df.show(truncate=False)

## 3 · `inferSchema` vs explicit `StructType`

| | `inferSchema=True` | Explicit `StructType` |
|---|---|---|
| Setup effort | Zero | You write the schema |
| Correctness on new data | **Silently wrong** — a decimal turns an int column into double | Always correct |
| Schema drift | Absorbed silently | Caught immediately |
| Read speed | Extra scan to sample | One pass |
| Production verdict | Exploratory only | **Use this** |

**Caution:** With an explicit schema, values that do not fit become `null` under `PERMISSIVE`
mode (or raise under `FAILFAST`).

In [ ]:
orders_schema = StructType([
    StructField("order_id",    StringType(),  True),
    StructField("customer_id", StringType(),  True),
    StructField("region",      StringType(),  True),
    StructField("amount",      DoubleType(), True),
    StructField("quantity",    IntegerType(), True),
])

orders = (
    spark.read
    .option("header", True)
    .option("mode", "PERMISSIVE")
    .schema(orders_schema)
    .csv(f"{DATA_DIR}/orders.csv")
)
print("== explicit schema ==")
orders.printSchema()
orders.show(truncate=False)

# The dirty rows became null rather than poisoning the column type.
orders.filter(F.col("amount").isNull()).show(truncate=False)

## 4 · Persisting a schema with `schema.json`

A schema registry is usually a JSON file in GCS or a BigQuery table. `StructType.fromJson`
rehydrates the `StructType` at runtime, so the contract is versioned independently of the
code.

In [ ]:
schema_json = orders_schema.json()
print(schema_json)

with open(f"{DATA_DIR}/orders_schema.json", "w", encoding="utf-8") as fh:
    fh.write(schema_json)

reloaded = StructType.fromJson(json.loads(schema_json))
print("\nreloaded schema:")
print(reloaded.simpleString())
print("\nreloaded == original:", reloaded == orders_schema)

In [ ]:
# Read using the persisted schema contract.
with open(f"{DATA_DIR}/orders_schema.json", "r", encoding="utf-8") as fh:
    schema_from_registry = StructType.fromJson(json.load(fh))

orders_from_registry = (
    spark.read.option("header", True).schema(schema_from_registry).csv(f"{DATA_DIR}/orders.csv")
)
print("rows read with the registry schema:", orders_from_registry.count())

## 5 · Inspecting schema and data

- `printSchema()` — the schema tree.
- `dtypes` — `(name, type)` pairs.
- `schema.json()` — the JSON contract.
- `show(truncate=False)` — never truncate strings when debugging.
- `describe()` — summary statistics for numeric columns.
- `columns` — column names.

In [ ]:
print("dtypes:")
print(orders.dtypes)
print("\ncolumns:", orders.columns)
print("\n== show(truncate=False) ==")
orders.show(truncate=False)
print("\n== describe() ==")
orders.describe().show(truncate=False)

## 6 · `select`, `withColumn`, and casting

`select` **replaces** the projection; `withColumn` **adds to** it. Mixing them up drops
columns silently.

Type names are SQL names: `"int"`, `"long"`, `"double"`, `"decimal(18,2)"`, `"string"`,
`"boolean"`. The Scala-style `"int_"` does not exist in Spark SQL.

In [ ]:
enriched = (
    orders
    .withColumn("amount_x2", F.col("amount") * F.lit(2.0))
    .withColumn("region_label", F.upper(F.col("region")))
    .withColumn("amount_int", F.col("amount").cast("int"))
    .withColumn("quantity_long", F.col("quantity").cast("long"))
    .withColumn("region_code", F.col("region").cast("string"))
)
enriched.show(truncate=False)

# select replaces the projection:
print("select only two columns:")
orders.select("order_id", "amount").show(truncate=False)

## 7 · `try_cast` — the Spark 3.5 safe-cast pattern

Plain `cast` is permissive by default: invalid values become `null` and you cannot tell
"the source said null" from "the source said `abc`". Spark 3.5 added `try_cast` /
`try_to_timestamp`, which fail to null **by design**.

The portable form is `F.expr("try_cast(col AS int)")` — the SQL function is guaranteed in
Spark 3.5, while the Python wrapper `F.try_cast` is not present in every build.

In [ ]:
raw_text = spark.createDataFrame(
    [("100",), ("abc",), ("",), ("-42",)],
    schema="amount_text string",
)
raw_text.withColumn("amount_plain", F.col("amount_text").cast("int")) \
        .withColumn("amount_try", F.expr("try_cast(amount_text AS int)")) \
        .show(truncate=False)

# Same trick for timestamps.
raw_text.withColumn("ts_try", F.expr("try_cast(amount_text AS timestamp)")).show(truncate=False)

print("Spark version:", spark.version, "-> try_cast available since 3.5")

## 8 · Null handling

| Function | Behaviour |
|---|---|
| `F.coalesce(a, b, c)` | First non-null value. SQL-standard, works on any types. |
| `F.nvl(a, b)` | `b` if `a` is null; `b` must be castable to `a`'s type. |
| `df.fillna(value_or_dict)` | Replace nulls column-wise. |
| `df.dropna(subset=[...])` | Drop rows null in the listed columns only. |
| `df.coalesce(n)` | **Partition-level** coalesce (fewer output files) — different meaning from `F.coalesce`. |

**Caution:** `F.sum`/`F.avg` ignore nulls, `F.count("*")` counts all rows, and `F.count("col")`
counts only non-null values. That difference is the classic reconciliation bug.

In [ ]:
print("== null counts ==")
orders.select(
    F.count("*").alias("rows"),
    F.count("amount").alias("non_null_amount"),
    F.countDistinct("customer_id").alias("customers"),
).show()

print("== dropna ==")
print("rows after dropna:", orders.dropna(subset=["amount"]).count())
orders.dropna(subset=["amount"]).show(truncate=False)

print("== fillna ==")
orders.fillna({"region": "UNKNOWN", "quantity": 0}).show(truncate=False)

print("== F.coalesce (row level) ==")
orders.withColumn("amount_effective", F.coalesce(F.col("amount"), F.lit(0.0))).show(truncate=False)

## 9 · Getting data to the driver

| Method | Driver memory | Use |
|---|---|---|
| `collect()` | Entire result | Tiny results only |
| `toPandas()` | Entire result | Sampling/inspection, with Arrow enabled |
| `toLocalIterator()` | One partition at a time | Row-by-row Python processing |
| `df.limit(n).toPandas()` | `n` rows | **The safe pandas pattern in Spark 3.5** |

**Caution:** `toPandas(n=...)` does **not** exist in Spark 3.5 — that argument arrived in Spark 4.0.
Bound the result with `limit()` or `sample()` first.

**Caution:** `toLocalIterator()` runs one job per partition: memory-safe but latency-heavy.

In [ ]:
# The safe pandas pattern: bound first, then convert.
small = orders.limit(3).toPandas()
print(type(small), small.shape)
print(small)
print("\ndtypes:")
print(small.dtypes)

# Memory-safe row-by-row iteration.
print("\ntoLocalIterator:")
for row in orders.select("order_id", "amount").toLocalIterator():
    print("  ", row["order_id"], row["amount"])

# The unsafe pattern, for tiny data only.
print("\ncollect (whole result):", orders.select("order_id").collect())

## 10 · Parquet round-trip

Parquet stores the schema in its footer, which makes it fast to scan and allows column
pruning and predicate pushdown. Convert once, then never touch CSV again.

In [ ]:
parquet_path = f"{DATA_DIR}/orders_parquet"
orders.write.mode("overwrite").option("compression", "snappy").parquet(parquet_path)

back = spark.read.parquet(parquet_path)
print("== parquet schema ==")
back.printSchema()
print("\nrows:", back.count())

# Column pruning and predicate pushdown show up in the plan.
back.filter(F.col("region") == "EMEA").select("order_id", "amount").explain(mode="formatted")

## 🔑 Key Takeaways

- `spark.read` is the unified reader; the shortcuts are sugar over `format(...).option(...).load(...)`.
- **Use explicit `StructType` schemas in production.** Inference samples, can infer `null` types, and silently changes types when new data arrives.
- Persist a schema with `schema.json()` and rehydrate it with `StructType.fromJson` — the contract becomes versioned data.
- `select` replaces the projection; `withColumn` adds to it. Confusing them drops columns silently.
- Use SQL type names for casts (`"int"`, `"double"`, `"decimal(18,2)"`) — `"int_"` is not a Spark SQL type.
- **Spark 3.5 `try_cast` via `F.expr("try_cast(col AS int)")`** is the safe cast: failures become null by design, not by accident.
- `F.coalesce` (row level) and `df.coalesce(n)` (partition level) are different functions with the same name.
- `count("*")` counts rows, `count("col")` counts non-null values, and `sum`/`avg` ignore nulls.
- `toPandas()` pulls everything to the driver — bound it with `limit()` first. `toLocalIterator()` is the memory-safe alternative.

---

## 📚 Next steps

- Read `orders.csv` with `FAILFAST` mode and compare the failure with `PERMISSIVE`.
- Continue to `04-transformations-and-joins.ipynb` for joins, windows, and aggregation.
